## Overview 

Below are the steps taken to consolidate and clean [USDA SNAP data](https://www.fns.usda.gov/pd/supplemental-nutrition-assistance-program-snap). In its original form, the monthly SNAP participation data by household/participant for all U.S. states and territories are separated by region across different Excel tabs in individual Excel sheets per year. For visualization purposes, I've created a few variations of SNAP datasets that contain the USDA data in a more accessible format. 

**Notes:**

* I only filtered the data for years 2018-2026 to show pre-, during, and post-COVID-19 shocks (my specific project).
* The specific data table I used from the USDA site was the ["National and/or State Level Monthly and/or Annual Data"](https://www.fns.usda.gov/sites/default/files/resource-files/snap-zip-fy69tocurrent-4.zip) zip file, which is linked (you will need to download this data).
* I only processed data for the 50 U.S. states + D.C., which is what I used for my maps/plots.
* The main driver of this was learning how to use the [Pandas Read_Excel](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) function, which is also linked here. It was beyond helpful and saved a lot of time. 

I completed this data cleaning for a final project, so do with that what you will...I'm sure there are more efficient ways to do this, but feel free to use my work as a template!

## Prepare Environment

In [1]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/Data/USDA_Data/snap-zip-fy69tocurrent-3/FY20.xlsx" # your file path goes there
xls = pd.ExcelFile(file)

print(xls.sheet_names)

['NERO', 'MARO', 'SERO', 'MWRO', 'SWRO', 'MPRO', 'WRO', 'US Summary']


In [ ]:
# Visualize what one sheet tab looks like 
df = pd.read_excel(file, sheet_name="NERO")
print(df.head(10))

## Begin Cleaning the Sheet(s)

In [ ]:
dfs = [] # make an empty list 

for sheet in xls.sheet_names:
    if "us summary" in sheet.lower().strip(): # skip the national/regional summary sheet included,
        continue

    df = pd.read_excel(file, sheet_name=sheet, skiprows=7) # USE PANDAS READ_EXCEL
    df["region"] = sheet # Creates region column
    dfs.append(df)

raw_df = pd.concat(dfs, ignore_index=True)

print("\n--- SHEET ---", sheet)
print(df.head(5))
print(df.columns)

In [ ]:
# Preview all sheets 
for sheet in xls.sheet_names:
    if "us summary" in sheet.lower():
        continue

    df = pd.read_excel(file, sheet_name=sheet, skiprows=7)

    print("\n---", sheet, "---")
    print(df.head(2))
    print(df.columns)

In [ ]:
# Build loop to clean data 
dfs = []

for sheet in xls.sheet_names:

    if "us summary" in sheet.lower():
        continue

    df = pd.read_excel(file, sheet_name=sheet, skiprows=7)

    df = df.dropna(how="all")

    # remove totals
    df = df[~df.iloc[:, 0].astype(str).str.contains("Total", na=False)]

    # rename columns FIRST (important!)
    df.columns = ["state_raw", "month", "households", "persons", "x1", "x2"]

    # detect state rows (ONLY rows where numeric data is missing)
    is_state_row = df["households"].isna() & df["persons"].isna()

    # assign state only where it is actually a state
    df.loc[is_state_row, "state"] = df.loc[is_state_row, "state_raw"]

    # MANUAL FIX for first block (Connecticut)
    df.loc[0:11, "state"] = "Connecticut"

    # forward fill state correctly
    df["state"] = df["state"].ffill()

    # remove state-only rows
    df = df[~is_state_row]

    # keep clean structure
    df = df[["state", "month", "households", "persons"]]

    df["region"] = sheet

    dfs.append(df)

raw_df = pd.concat(dfs, ignore_index=True)

In [ ]:
# Preview
raw_df.head(20)

In [ ]:
# Coninute testing 
dfs = []

for sheet in xls.sheet_names:

    if "us summary" in sheet.lower():
        continue

    df = pd.read_excel(file, sheet_name=sheet, skiprows=3)

    df = df.dropna(how="all")

    # remove top 3 junk rows 
    df = df.iloc[3:].reset_index(drop=True) 

    # FIX: keep only needed columns
    df = df.iloc[:, :3]

    # now safe to rename
    df.columns = ["col0", "households", "persons"]

    state = None
    cleaned_rows = []

    for _, row in df.iterrows():

        col0 = str(row["col0"]).strip()

        # detect state row
        if pd.isna(row["households"]) and pd.isna(row["persons"]) and col0 not in ["Total", "Fiscal Year and Month"]:
            state = col0
            continue

        # skip totals
        if col0 == "Total":
            continue

        cleaned_rows.append([
            state,
            col0,
            row["households"],
            row["persons"]
        ])

    clean_df = pd.DataFrame(cleaned_rows, columns=["state", "month", "households", "persons"])

    clean_df["region"] = sheet

    dfs.append(clean_df)

raw_df = pd.concat(dfs, ignore_index=True)

In [ ]:
raw_df.head(20)

In [ ]:
# see most of the data
raw_df.tail(20)

In [ ]:
# There should be 12 rows for 51 one entities...
raw_df.shape

In [ ]:
raw_df["state"].unique()

In [ ]:
raw_df["state"].value_counts().head(20)

In [ ]:
# Remove rows totaling regions
region_codes = ["NERO", "MARO", "SERO", "MWRO", "SWRO", "MPRO", "WRO"]
raw_df = raw_df[~raw_df["state"].isin(region_codes)]
raw_df = raw_df[~raw_df["state"].str.contains("RO$", na=False)]
raw_df["state"].unique()

In [ ]:
# For mapping this is more standard 
keep_states = [
    "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut",
    "Delaware","Florida","Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa",
    "Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts","Michigan",
    "Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada","New Hampshire",
    "New Jersey","New Mexico","New York","North Carolina","North Dakota","Ohio",
    "Oklahoma","Oregon","Pennsylvania","Rhode Island","South Carolina","South Dakota",
    "Tennessee","Texas","Utah","Vermont","Virginia","Washington","West Virginia",
    "Wisconsin","Wyoming","District of Columbia"
]

raw_df = raw_df[raw_df["state"].isin(keep_states)]

In [ ]:
raw_df["year"] = 2020 # add year column
df_2020 = raw_df
df_2020.head(20)

## Repeat For All Years of Data

In [ ]:
# Base template to clean all sheets/per year file --> Basically creating a function to run each sheet through and clean like we did above

import pandas as pd
import re

def clean_snap_file(file_path, xls):

    dfs = []

    # STEP A: automatically extract year from filename
    year = re.search(r"FY(\d{2})", file_path).group(1)
    year = "20" + year   # FY20 → 2020

    for sheet in xls.sheet_names:

        if "us summary" in sheet.lower():
            continue

        df = pd.read_excel(file_path, sheet_name=sheet, skiprows=3)

        # remove top 3 junk rows  (for some reason, the skiprows does not skip all rows)
        df = df.iloc[3:].reset_index(drop=True)

        df = df.dropna(how="all")

        df = df.iloc[:, :3]

        df.columns = ["col0", "households", "persons"]

        state = None
        cleaned_rows = []

        for _, row in df.iterrows():

            col0 = str(row["col0"]).strip()

            # detect state header rows
            if pd.isna(row["households"]) and pd.isna(row["persons"]) and col0 != "Total":
                state = col0
                continue

            # skip totals
            if col0 == "Total":
                continue

            cleaned_rows.append([
                state,
                col0,
                row["households"],
                row["persons"],
                sheet,
                year
            ])

        clean_df = pd.DataFrame(
            cleaned_rows,
            columns=["state", "month", "households", "persons", "region", "year"]
        )

        dfs.append(clean_df)

    raw_df = pd.concat(dfs, ignore_index=True)

    # KEEP ONLY 50 STATES + DC
    keep_states = [
        "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut",
        "Delaware","Florida","Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa",
        "Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts","Michigan",
        "Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada","New Hampshire",
        "New Jersey","New Mexico","New York","North Carolina","North Dakota","Ohio",
        "Oklahoma","Oregon","Pennsylvania","Rhode Island","South Carolina","South Dakota",
        "Tennessee","Texas","Utah","Vermont","Virginia","Washington","West Virginia",
        "Wisconsin","Wyoming","District of Columbia"
    ]

    raw_df = raw_df[raw_df["state"].isin(keep_states)]

    return raw_df

In [ ]:
# Import for older excel files
!pip install xlrd

In [ ]:
# For 2018
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/snap-zip-fy69tocurrent-3/FY18.xls"   # ONLY CHANGE THIS LINE W YOUR PATH
xls = pd.ExcelFile(file)
df_2018 = clean_snap_file(file, xls)

In [ ]:
df_2018.head()

In [ ]:
# For 2019
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/snap-zip-fy69tocurrent-3/FY19.xls"   
xls = pd.ExcelFile(file)
df_2019 = clean_snap_file(file, xls)

In [ ]:
df_2019.head()

In [ ]:
# For 2021
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY21.xlsx"   # ONLY CHANGE THIS LINE W YOUR PATH
xls = pd.ExcelFile(file)
df_2021 = clean_snap_file(file, xls)

In [ ]:
df_2021.head()

In [ ]:
# For 2022
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY22.xlsx"
xls = pd.ExcelFile(file)
df_2022 = clean_snap_file(file, xls)
df_2022.head()

In [ ]:
# For 2023
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY23.xlsx"
xls = pd.ExcelFile(file)
df_2023 = clean_snap_file(file, xls)
df_2023.head()

In [ ]:
# For 2024
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY24.xlsx"
xls = pd.ExcelFile(file)
df_2024 = clean_snap_file(file, xls)
df_2024.head()

In [ ]:
# For 2025
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY25.xlsx"
xls = pd.ExcelFile(file)
df_2025 = clean_snap_file(file, xls)
df_2025.head()

In [ ]:
# For 2026
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY26.xlsx"
xls = pd.ExcelFile(file)
df_2026 = clean_snap_file(file, xls)
df_2026.head()

In [ ]:
# Create individual CSV's to store and have for each year 
df_2018.to_csv("snap_2018_clean.csv", index=False)
df_2019.to_csv("snap_2019_clean.csv", index=False)
df_2020.to_csv("snap_2020_clean.csv", index=False)
df_2021.to_csv("snap_2021_clean.csv", index=False)
df_2022.to_csv("snap_2022_clean.csv", index=False)
df_2023.to_csv("snap_2023_clean.csv", index=False)
df_2024.to_csv("snap_2024_clean.csv", index=False)
df_2025.to_csv("snap_2025_clean.csv", index=False)
df_2026.to_csv("snap_2026_clean.csv", index=False)

## Merge into Core Dataset

In [ ]:
# Putting this function here to prevent type errors when playing with the whole dataset 
import pandas as pd
import re

def clean_numeric(df):
    for col in ["persons", "households"]:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

In [ ]:
whole_df = pd.concat([
    df_2018,
    df_2019,
    df_2020,
    df_2021,
    df_2022,
    df_2023,
    df_2024,
    df_2025,
    df_2026
], ignore_index=True)

print(whole_df.shape)
print(whole_df["year"].unique())
print(whole_df["state"].nunique())

In [ ]:
# Use the above fucntion 
whole_df = clean_numeric(whole_df)

In [ ]:
# Clean month column 
whole_df["month"] = (
    whole_df["month"]
    .astype(str)
    .str.replace(r"/\d+", "", regex=True)   # removes /2, /1, etc.
    .str.strip()
)

In [ ]:
# This is a big Panel Dataset you can play with!
whole_df["date"] = pd.to_datetime(
    whole_df["month"],
    format="%b %Y",
    errors="coerce"
)

whole_df.to_csv("snap_panel_2018_2026.csv", index=False)
whole_df.head(20)

Here is the core panel dataset that is the basis of my research. From here, you can go a lot of ways with your research because you have every month of participation data for all 50 states (+ DC) from 2018-2026. It is not the most friendly format, but this is just the foundational dataset that this data cleaning notebook attempted to create for future use. 

## Aggregate Data

In [ ]:
# Confirm Numeric
whole_df["households"] = pd.to_numeric(whole_df["households"], errors="coerce")
whole_df["persons"] = pd.to_numeric(whole_df["persons"], errors="coerce")

# Extract Year 
whole_df["year"] = whole_df["date"].dt.year

In [86]:
# aggregate 
yearly_df = (
    whole_df
    .groupby(["state", "year"])
    .agg({
        "households": "mean",
        "persons": "mean"
    })
    .reset_index()
)

yearly_df.head(20)
shock_pivot.to_csv("snap_yearly_states.csv", index=False)

,state,year,households,persons
0,Alabama,2018,350620.000000,743843.333333
1,Alabama,2019,308935.190476,652029.142857
2,Alabama,2020,359078.250000,747437.750000
3,Alabama,2021,373708.916667,764978.250000
4,Alabama,2022,381479.750000,765263.166667
5,Alabama,2023,387946.750000,769893.750000
6,Alabama,2024,375779.250000,751877.250000
7,Alabama,2025,367461.090909,732122.181818
8,Alabama,2026,NaN,NaN
9,Alaska,2018,36065.333333,80102.000000


Above is an aggregated, yearly SNAP participation dataset for all states + DC.

### COVID-Specific Dataset Edits

In [87]:
def covid_period(year):
    if year <= 2019:
        return "Pre"
    elif year <= 2021:
        return "Shock"
    elif year <= 2023:
        return "Recovery"
    else:
        return "Post"

In [91]:
yearly_df["period"] = yearly_df["year"].apply(covid_period)
yearly_df.head(20)

,state,year,households,persons,period
0,Alabama,2018,350620.000000,743843.333333,Pre
1,Alabama,2019,308935.190476,652029.142857,Pre
2,Alabama,2020,359078.250000,747437.750000,Shock
3,Alabama,2021,373708.916667,764978.250000,Shock
4,Alabama,2022,381479.750000,765263.166667,Recovery
5,Alabama,2023,387946.750000,769893.750000,Recovery
6,Alabama,2024,375779.250000,751877.250000,Post
7,Alabama,2025,367461.090909,732122.181818,Post
8,Alabama,2026,NaN,NaN,Post
9,Alaska,2018,36065.333333,80102.000000,Pre


In [92]:
# State level COVID-19 Impact window 
shock_summary = (
    yearly_df
    .groupby(["state", "period"])
    .agg({
        "persons": "mean"
    })
    .reset_index()
)
shock_summary.head()

,state,period,persons
0,Alabama,Post,741999.715909
1,Alabama,Pre,697936.238095
2,Alabama,Recovery,767578.458333
3,Alabama,Shock,756208.000000
4,Alaska,Post,68254.200758


In [94]:
# Pivot to see period changes 
shock_pivot = shock_summary.pivot(
    index="state",
    columns="period",
    values="persons"
).reset_index()
shock_pivot.head(20)

period,state,Post,Pre,Recovery,Shock
0,Alabama,7.419997e+05,6.979362e+05,7.675785e+05,7.562080e+05
1,Alaska,6.825420e+04,7.932317e+04,6.807854e+04,8.477350e+04
2,Arizona,8.856886e+05,7.714121e+05,8.799055e+05,8.555402e+05
3,Arkansas,2.394816e+05,3.577722e+05,2.560158e+05,3.550328e+05
4,California,5.431675e+06,3.663049e+06,5.024422e+06,4.375347e+06
5,Colorado,6.017016e+05,4.328820e+05,5.581426e+05,5.095589e+05
6,Connecticut,3.748499e+05,3.541241e+05,3.869744e+05,3.725637e+05
7,Delaware,1.184876e+05,1.252587e+05,1.202899e+05,1.175870e+05
8,District of Columbia,1.395686e+05,1.067724e+05,1.412374e+05,1.325357e+05
9,Florida,2.952352e+06,2.784358e+06,2.978840e+06,3.286776e+06


Something to note: Each period value = average monthly SNAP participation over multiple years

The above table represents "Average monthly SNAP participation during Pre / Shock / Recovery" not total SNAP people impacted during COVID-19. Also this is "Persons" participation, not "Households."

In [95]:
# Create Shock effect column (Covid_pct_change)
shock_pivot["covid_pct_change"] = (
    (shock_pivot["Shock"] - shock_pivot["Pre"])
    / shock_pivot["Pre"]
)
shock_pivot.head(10)

period,state,Post,Pre,Recovery,Shock,covid_pct_change
0,Alabama,7.419997e+05,6.979362e+05,7.675785e+05,7.562080e+05,0.083492
1,Alaska,6.825420e+04,7.932317e+04,6.807854e+04,8.477350e+04,0.068710
2,Arizona,8.856886e+05,7.714121e+05,8.799055e+05,8.555402e+05,0.109057
3,Arkansas,2.394816e+05,3.577722e+05,2.560158e+05,3.550328e+05,-0.007657
4,California,5.431675e+06,3.663049e+06,5.024422e+06,4.375347e+06,0.194455
5,Colorado,6.017016e+05,4.328820e+05,5.581426e+05,5.095589e+05,0.177131
6,Connecticut,3.748499e+05,3.541241e+05,3.869744e+05,3.725637e+05,0.052071
7,Delaware,1.184876e+05,1.252587e+05,1.202899e+05,1.175870e+05,-0.061247
8,District of Columbia,1.395686e+05,1.067724e+05,1.412374e+05,1.325357e+05,0.241292
9,Florida,2.952352e+06,2.784358e+06,2.978840e+06,3.286776e+06,0.180443


In [96]:
# Create Covid Index to normalize
shock_pivot["covid_index"] = shock_pivot["Shock"] / shock_pivot["Pre"]
shock_pivot.head(10)

period,state,Post,Pre,Recovery,Shock,covid_pct_change,covid_index
0,Alabama,7.419997e+05,6.979362e+05,7.675785e+05,7.562080e+05,0.083492,1.083492
1,Alaska,6.825420e+04,7.932317e+04,6.807854e+04,8.477350e+04,0.068710,1.068710
2,Arizona,8.856886e+05,7.714121e+05,8.799055e+05,8.555402e+05,0.109057,1.109057
3,Arkansas,2.394816e+05,3.577722e+05,2.560158e+05,3.550328e+05,-0.007657,0.992343
4,California,5.431675e+06,3.663049e+06,5.024422e+06,4.375347e+06,0.194455,1.194455
5,Colorado,6.017016e+05,4.328820e+05,5.581426e+05,5.095589e+05,0.177131,1.177131
6,Connecticut,3.748499e+05,3.541241e+05,3.869744e+05,3.725637e+05,0.052071,1.052071
7,Delaware,1.184876e+05,1.252587e+05,1.202899e+05,1.175870e+05,-0.061247,0.938753
8,District of Columbia,1.395686e+05,1.067724e+05,1.412374e+05,1.325357e+05,0.241292,1.241292
9,Florida,2.952352e+06,2.784358e+06,2.978840e+06,3.286776e+06,0.180443,1.180443


In [108]:
# Reorder 

# ensure it's clean
shock_pivot = shock_pivot.reset_index()

# drop if it exists
shock_pivot = shock_pivot.drop(columns=["period"], errors="ignore")

shock_pivot = shock_pivot[[
    "state",
    "Pre",
    "Shock",
    "Recovery",
    "Post",
    "covid_pct_change",
    "covid_index",
]]
shock_pivot.head(10)
shock_pivot.to_csv("snap_covid_shock_states.csv", index=False)

How to interpret COVID_Index:
* If value = 1.00
→ no change at all
* If value > 1.00
→ SNAP increased during COVID
* If value = 1.20
→ SNAP was 20% higher than pre-COVID baseline
* If value = 0.95
→ SNAP slightly decreased

Note: I could not figure out how to get rid of this period column, which I think has just become a horizontal header 

This is the more accessible dataset I needed for my research. My project using this data is also linked on my GitHub Repo!